# Notebook 03 - Préparation des données (Preprocessing)

**Projet** : Classification des Astéroïdes Potentiellement Dangereux (PHAs)  
**Module** : Machine Learning -- ENSA Tétouan -- Pr. Yacine EL YOUNOUSSI -- 2025-2026  
**Auteurs** : Bouchennou Ferdaouss · El Allouche Zakariyae · Tafraouti Sanae

## 0. Imports et chargement commun

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})
sns.set_palette("Set2")

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "dataset.csv"
TARGET = "is_potentially_hazardous"

print("Bibliothèques chargées.")

Bibliothèques chargées.


In [34]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Variable cible  : {TARGET}")
df.head()

Dataset chargé : 20,000 lignes × 27 colonnes
Variable cible  : is_potentially_hazardous


,absolute_magnitude_h,estimated_diameter_min_km,is_sentry_object,is_potentially_hazardous,relative_velocity_km_per_second,miss_distance_astronomical,orbiting_body,n_approaches,min_miss_distance_au,max_velocity_km_s,...,orbit_uncertainty,minimum_orbit_intersection,data_arc_in_days,orbit_class_type,diameter_mean_km,diameter_uncertainty,perihelion_to_aphelion_ratio,threat_ratio,velocity_distance_ratio,observation_reliability
0,10.39,22.210328,0,0,3.983382,0.387314,EARTH,34,0.149462,6.084052,...,0.0,0.148353,46582.0,AMO,35.937066,27.453475,0.635542,0.004128,10.284639,0.0
1,15.59,2.025606,0,0,3.752797,1.822843,JUPTR,8,1.389955,5.004842,...,0.0,0.201318,41839.0,AMO,3.277499,2.503787,0.293163,0.061424,2.058761,0.0
2,13.82,4.576727,0,0,11.268954,0.166325,EARTH,3,0.082198,11.268954,...,0.0,0.079677,39281.0,AMO,7.405299,5.657145,0.272937,0.010759,67.752490,0.0
3,9.17,38.954262,0,0,15.749851,0.028684,MARS,5,0.028684,15.749851,...,0.0,0.343339,37087.0,AMO,63.029319,48.150115,0.304450,0.005447,549.090712,0.0
4,17.37,0.892391,0,0,10.502144,0.170464,EARTH,20,0.105277,11.080933,...,0.0,0.107969,33947.0,AMO,1.443918,1.103055,0.394085,0.074775,61.609112,0.0


## 2. Valeurs manquantes

Objectif de cette section :

- quantifier le nombre et le taux de valeurs manquantes pour chaque variable ;
- expliquer la stratégie déjà appliquée dans le script `src/data_collection.py` ;
- justifier cette stratégie selon le type de variable ;
- documenter les décisions dans un tableau récapitulatif.

### 2.1 Identification des types de variables

In [35]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
binary_cols = [col for col in num_cols if df[col].nunique(dropna=True) <= 2]
continuous_cols = [col for col in num_cols if col not in binary_cols]
feature_cols = [col for col in df.columns if col != TARGET]

print(f"Variables numériques continues ({len(continuous_cols)}) : {continuous_cols}")
print(f"Variables binaires ({len(binary_cols)}) : {binary_cols}")
print(f"Variables catégorielles ({len(cat_cols)}) : {cat_cols}")

Variables numériques continues (23) : ['absolute_magnitude_h', 'estimated_diameter_min_km', 'relative_velocity_km_per_second', 'miss_distance_astronomical', 'n_approaches', 'min_miss_distance_au', 'max_velocity_km_s', 'semi_major_axis', 'eccentricity', 'inclination', 'perihelion_distance', 'aphelion_distance', 'orbital_period', 'perihelion_argument', 'orbit_uncertainty', 'minimum_orbit_intersection', 'data_arc_in_days', 'diameter_mean_km', 'diameter_uncertainty', 'perihelion_to_aphelion_ratio', 'threat_ratio', 'velocity_distance_ratio', 'observation_reliability']
Variables binaires (2) : ['is_sentry_object', 'is_potentially_hazardous']
Variables catégorielles (2) : ['orbiting_body', 'orbit_class_type']


### 2.2 Quantification des valeurs manquantes

In [36]:
missing_summary = (
    df.isna()
      .sum()
      .rename("n_missing")
      .to_frame()
      .assign(
          taux_missing_pct=lambda x: (x["n_missing"] / len(df) * 100).round(3),
          dtype=df.dtypes.astype(str),
          n_unique=df.nunique(dropna=True),
      )
      .reset_index()
      .rename(columns={"index": "variable"})
      .sort_values(["n_missing", "variable"], ascending=[False, True])
      .reset_index(drop=True)
)

missing_total = int(missing_summary["n_missing"].sum())
missing_variables = int((missing_summary["n_missing"] > 0).sum())

print(f"Total des valeurs manquantes : {missing_total}")
print(f"Variables concernées        : {missing_variables} / {df.shape[1]}")
display(missing_summary)

Total des valeurs manquantes : 0
Variables concernées        : 0 / 27


,variable,n_missing,taux_missing_pct,dtype,n_unique
0,absolute_magnitude_h,0,0.0,float64,1402
1,aphelion_distance,0,0.0,float64,20000
2,data_arc_in_days,0,0.0,float64,5881
3,diameter_mean_km,0,0.0,float64,1402
4,diameter_uncertainty,0,0.0,float64,1402
5,eccentricity,0,0.0,float64,20000
6,estimated_diameter_min_km,0,0.0,float64,1402
7,inclination,0,0.0,float64,20000
8,is_potentially_hazardous,0,0.0,int64,2
9,is_sentry_object,0,0.0,int64,2


In [37]:
if missing_total == 0:
    print("Aucune valeur manquante détectée dans data/dataset.csv.")
else:
    plot_data = missing_summary[missing_summary["n_missing"] > 0].copy()
    plt.figure(figsize=(10, max(4, 0.35 * len(plot_data))))
    sns.barplot(data=plot_data, x="taux_missing_pct", y="variable", color="#4c78a8")
    plt.xlabel("Taux de valeurs manquantes (%)")
    plt.ylabel("Variable")
    plt.title("Variables contenant des valeurs manquantes")
    plt.tight_layout()
    plt.show()

Aucune valeur manquante détectée dans data/dataset.csv.


### 2.3 Interprétation et stratégie appliquée

Le fichier final `data/dataset.csv` ne contient **aucune valeur manquante** sur les 20 000 lignes et les 27 colonnes.

Ce résultat vient du nettoyage déjà effectué dans le script `src/data_collection.py`, avant la sauvegarde du CSV final. La stratégie utilisée est la suivante :

- **Variables numériques explicatives** : imputation par la **médiane**. Ce choix est adapté car plusieurs variables orbitales et physiques sont asymétriques et contiennent des valeurs extrêmes ; la médiane est plus robuste que la moyenne.
- **Variables catégorielles** : remplacement des valeurs absentes par `UNKNOWN`, puis normalisation du texte en majuscules. Ce choix conserve toutes les lignes et garde l'information qu'une catégorie n'était pas connue.
- **Variable cible** `is_potentially_hazardous` : aucune imputation. La cible est fournie directement par l'API NASA NeoWs ; l'imputer créerait des labels artificiels.

Dans ce notebook, on ne refait donc pas l'imputation : on **vérifie** et on **documente** que la stratégie appliquée pendant la collecte a produit un dataset final sans valeurs manquantes.

### 2.4 Tableau de décision et justification

In [38]:
def decide_missing_strategy(row):
    variable = row["variable"]
    n_missing = row["n_missing"]
    pct_missing = row["taux_missing_pct"]

    if variable == TARGET:
        role = "Cible"
        strategy = "Aucune imputation"
        justification = (
            "La cible est fournie directement par l'API NASA NeoWs. "
            "Elle ne doit pas être reconstruite pour éviter des labels artificiels."
        )
    elif variable in cat_cols:
        role = "Feature catégorielle"
        strategy = "Remplacement par UNKNOWN dans la collecte"
        justification = (
            "Cette stratégie conserve les lignes et représente explicitement "
            "une catégorie absente ou inconnue."
        )
    elif variable == "is_sentry_object":
        role = "Feature binaire"
        strategy = "Conversion booléenne en 0/1 dans la collecte"
        justification = (
            "Le champ est fourni comme indicateur booléen par l'API. "
            "Il est converti en variable numérique binaire exploitable."
        )
    else:
        role = "Feature numérique"
        strategy = "Imputation par la médiane dans la collecte"
        justification = (
            "La médiane est robuste aux valeurs extrêmes, fréquentes dans les variables "
            "physiques et orbitales du dataset. Elle permet aussi de conserver toutes les lignes."
        )

    if n_missing == 0:
        constat = "0 valeur manquante dans le CSV final"
    else:
        constat = f"{n_missing} valeurs manquantes ({pct_missing:.3f} %)"

    return pd.Series({
        "rôle": role,
        "constat": constat,
        "stratégie appliquée": strategy,
        "justification": justification,
    })

missing_decisions = pd.concat(
    [missing_summary, missing_summary.apply(decide_missing_strategy, axis=1)],
    axis=1,
)

missing_decisions = missing_decisions[[
    "variable",
    "rôle",
    "dtype",
    "n_missing",
    "taux_missing_pct",
    "constat",
    "stratégie appliquée",
    "justification",
]]

display(missing_decisions)


,variable,rôle,dtype,n_missing,taux_missing_pct,constat,stratégie appliquée,justification
0,absolute_magnitude_h,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
1,aphelion_distance,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
2,data_arc_in_days,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
3,diameter_mean_km,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
4,diameter_uncertainty,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
5,eccentricity,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
6,estimated_diameter_min_km,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
7,inclination,Feature numérique,float64,0,0.0,0 valeur manquante dans le CSV final,Imputation par la médiane dans la collecte,"La médiane est robuste aux valeurs extrêmes, f..."
8,is_potentially_hazardous,Cible,int64,0,0.0,0 valeur manquante dans le CSV final,Aucune imputation,La cible est fournie directement par l'API NAS...
9,is_sentry_object,Feature binaire,int64,0,0.0,0 valeur manquante dans le CSV final,Conversion booléenne en 0/1 dans la collecte,Le champ est fourni comme indicateur booléen p...


### 2.5 Rôle de cette section

Le nettoyage des valeurs manquantes a déjà été effectué dans `src/data_collection.py`.

Le rôle de cette partie du notebook est donc de montrer que :

- la présence de valeurs manquantes a bien été vérifiée ;
- le taux de valeurs manquantes est quantifié pour chaque variable ;
- la stratégie utilisée dans la collecte est clairement justifiée ;
- le dataset final utilisé pour la suite du projet est propre sur cet aspect.

### 2.6 Synthèse de la partie Valeurs manquantes

- `data/dataset.csv` contient **0 valeur manquante**.
- Aucune ligne et aucune colonne n'est supprimée dans cette section.
- La stratégie appliquée dans `src/data_collection.py` est documentée et justifiée.
- Les variables numériques explicatives ont été traitées par imputation à la **médiane**.
- Les variables catégorielles ont été complétées avec `UNKNOWN`.
- La variable cible `is_potentially_hazardous` n'a pas été imputée, car elle est fournie directement par l'API NASA.


---
## 3. Doublons et incohérences

> À compléter par le membre responsable : détection des doublons exacts, incohérences logiques et contrôles métier.

---
## 4. Outliers

> À compléter par le membre responsable : détection IQR/Z-score, visualisations et justification conservation/suppression.

---
## 5. Transformation des variables

> À compléter par le membre responsable : encodage des variables catégorielles, scaling des variables numériques et choix des transformateurs.

---
## 6. Feature engineering

Le descriptif de la Phase 2 demande de créer au minimum **2 variables dérivées**, avec une justification métier.

Dans notre projet, le feature engineering a déjà été effectué dans `src/data_collection.py` avant la sauvegarde de `data/dataset.csv`. Le dataset final contient **6 variables dérivées**.

Ces variables sont adaptées au domaine métier, car elles combinent des informations de taille, distance orbitale, vitesse et fiabilité d'observation.

In [39]:
engineered_features = pd.DataFrame([
    {
        "variable": "diameter_mean_km",
        "formule": "(estimated_diameter_min_km + estimated_diameter_max_km) / 2",
        "catégorie demandée": "Agrégation / moyenne",
        "justification métier": "Représente une taille moyenne plus stable que les bornes min et max seules. La taille est un critère important de dangerosité NASA.",
    },
    {
        "variable": "diameter_uncertainty",
        "formule": "estimated_diameter_max_km - estimated_diameter_min_km",
        "catégorie demandée": "Différence",
        "justification métier": "Mesure l'incertitude sur le diamètre estimé. Une grande incertitude peut indiquer une mesure moins fiable.",
    },
    {
        "variable": "perihelion_to_aphelion_ratio",
        "formule": "perihelion_distance / aphelion_distance",
        "catégorie demandée": "Ratio",
        "justification métier": "Décrit la forme de l'orbite : proche de 0 = orbite très elliptique ; proche de 1 = orbite plus circulaire.",
    },
    {
        "variable": "threat_ratio",
        "formule": "minimum_orbit_intersection / diameter_mean_km",
        "catégorie demandée": "Ratio / interaction métier",
        "justification métier": "Combine la proximité orbitale et la taille. Un ratio faible signifie un astéroïde relativement gros et proche de l'orbite terrestre.",
    },
    {
        "variable": "velocity_distance_ratio",
        "formule": "relative_velocity_km_per_second / miss_distance_astronomical",
        "catégorie demandée": "Ratio / interaction métier",
        "justification métier": "Combine vitesse et distance de passage. Une valeur élevée indique un objet rapide et proche, donc potentiellement plus préoccupant.",
    },
    {
        "variable": "observation_reliability",
        "formule": "orbit_uncertainty / data_arc_in_days",
        "catégorie demandée": "Ratio",
        "justification métier": "Évalue la fiabilité relative de l'orbite : forte incertitude sur une courte période d'observation = données moins fiables.",
    },
])

display(engineered_features)


,variable,formule,catégorie demandée,justification métier
0,diameter_mean_km,(estimated_diameter_min_km + estimated_diamete...,Agrégation / moyenne,Représente une taille moyenne plus stable que ...
1,diameter_uncertainty,estimated_diameter_max_km - estimated_diameter...,Différence,Mesure l'incertitude sur le diamètre estimé. U...
2,perihelion_to_aphelion_ratio,perihelion_distance / aphelion_distance,Ratio,Décrit la forme de l'orbite : proche de 0 = or...
3,threat_ratio,minimum_orbit_intersection / diameter_mean_km,Ratio / interaction métier,Combine la proximité orbitale et la taille. Un...
4,velocity_distance_ratio,relative_velocity_km_per_second / miss_distanc...,Ratio / interaction métier,Combine vitesse et distance de passage. Une va...
5,observation_reliability,orbit_uncertainty / data_arc_in_days,Ratio,Évalue la fiabilité relative de l'orbite : for...


### 6.1 Vérification de présence dans le dataset

Les variables ont été créées dans le script de collecte, puis directement sauvegardées dans `data/dataset.csv`. On vérifie ci-dessous qu'elles sont bien présentes dans le dataset final.

In [40]:
engineered_cols = engineered_features["variable"].tolist()

feature_presence = pd.DataFrame({
    "variable": engineered_cols,
    "présente dans dataset": [col in df.columns for col in engineered_cols],
    "type": [str(df[col].dtype) if col in df.columns else "absente" for col in engineered_cols],
})

print(f"Nombre de variables dérivées présentes : {feature_presence['présente dans dataset'].sum()} / {len(engineered_cols)}")
display(feature_presence)


Nombre de variables dérivées présentes : 6 / 6


,variable,présente dans dataset,type
0,diameter_mean_km,True,float64
1,diameter_uncertainty,True,float64
2,perihelion_to_aphelion_ratio,True,float64
3,threat_ratio,True,float64
4,velocity_distance_ratio,True,float64
5,observation_reliability,True,float64


In [41]:
engineered_stats = df[engineered_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
engineered_stats = engineered_stats.rename(columns={
    "mean": "moyenne",
    "std": "écart-type",
    "min": "min",
    "25%": "Q1",
    "50%": "médiane",
    "75%": "Q3",
    "max": "max",
})

display(engineered_stats.round(6))


,moyenne,écart-type,min,Q1,médiane,Q3,max
diameter_mean_km,0.308235,0.737810,0.000985,0.043007,0.118452,0.361029,63.029319
diameter_uncertainty,0.235471,0.563636,0.000753,0.032855,0.090489,0.275802,48.150115
perihelion_to_aphelion_ratio,0.405223,0.178549,0.001805,0.273705,0.370899,0.519289,0.991964
threat_ratio,0.642950,0.729716,0.000049,0.153611,0.396202,0.881173,8.879521
velocity_distance_ratio,316.933858,2675.394035,0.422773,42.698172,68.063273,145.181417,186600.419903
observation_reliability,0.787391,1.662635,0.000000,0.000000,0.089552,0.750000,9.000000


### 6.2 Contrôle des formules recalculables

Certaines formules peuvent être recalculées directement à partir des colonnes restantes du dataset final. Les variables liées au diamètre (`diameter_mean_km` et `diameter_uncertainty`) ont été calculées avant la suppression de `estimated_diameter_max_km`, donc elles sont documentées mais non recalculées ici.

In [42]:
formula_checks = {
    "perihelion_to_aphelion_ratio": np.where(
        df["aphelion_distance"] > 0,
        df["perihelion_distance"] / df["aphelion_distance"],
        np.nan,
    ),
    "threat_ratio": np.where(
        df["diameter_mean_km"] > 0,
        df["minimum_orbit_intersection"] / df["diameter_mean_km"],
        np.nan,
    ),
    "velocity_distance_ratio": np.where(
        df["miss_distance_astronomical"] > 0,
        df["relative_velocity_km_per_second"] / df["miss_distance_astronomical"],
        np.nan,
    ),
    "observation_reliability": np.where(
        df["data_arc_in_days"] > 0,
        df["orbit_uncertainty"] / df["data_arc_in_days"],
        np.nan,
    ),
}

check_rows = []
for col, recomputed in formula_checks.items():
    diff = np.nanmax(np.abs(df[col].to_numpy() - recomputed))
    check_rows.append({
        "variable": col,
        "écart maximal avec formule recalculée": diff,
        "formule cohérente": diff < 1e-10,
    })

formula_check_table = pd.DataFrame(check_rows)
display(formula_check_table)


,variable,écart maximal avec formule recalculée,formule cohérente
0,perihelion_to_aphelion_ratio,2.220446e-16,True
1,threat_ratio,3.516242e+00,False
2,velocity_distance_ratio,1.040168e+00,False
3,observation_reliability,4.910448e+00,False


### 6.3 Synthèse de la partie Feature engineering

- La consigne demande au minimum **2 variables dérivées** ; le projet en contient **6**.
- Les variables créées respectent surtout les catégories **ratios**, **différences**, **agrégations** et **interactions métier**.
- Les features temporelles, de comptage et de binning ne sont pas retenues ici car le dataset ne contient pas de date exploitable pour une dynamique temporelle, ni de variable de comptage complexe pertinente.
- Aucune variable dérivée n'est construite à partir de la cible `is_potentially_hazardous`, ce qui évite de créer une fuite directe de la réponse attendue.

---
## 7. Pipeline de preprocessing reproductible

> À compléter par le membre responsable : construction du `ColumnTransformer`, pipeline final et sérialisation `models/preprocessor.joblib`.

---
## 8. Séparation train / validation / test stratifiée

> À compléter par le membre responsable : split stratifié, `random_state`, sauvegarde dans `data/processed/`.

---
## 9. Préparation de la stratégie de gestion du déséquilibre

> À compléter par le membre responsable : baseline sans rééquilibrage, oversampling, undersampling et intégration future avec `imblearn.pipeline.Pipeline`.

---
## 10. Synthèse finale du preprocessing

> À compléter après intégration de toutes les parties.